In [ ]:
# RECOVERY CELL R1:

import subprocess
import sys

for pkg in ['pretty_midi', 'miditok', 'tqdm']:
    subprocess.run([sys.executable, '-m', 'pip', 'install', pkg, '-q'])

# Imports
import os, glob, json, pickle, random, warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import pretty_midi
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import Adam
from torch.optim.lr_scheduler import ReduceLROnPlateau
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from tqdm import tqdm
from collections import Counter

# Seed
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Mount Drive
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

print(f" Libraries ready | Device: {DEVICE}")

Mounted at /content/drive
 Libraries ready | Device: cuda


In [ ]:
# RECOVERY CELL R2

# ── Paths ─────────────────────────────────────────────────
MAESTRO_ROOT   = '/content/drive/MyDrive/maestro-v3.0.0'
CSV_PATH       = os.path.join(MAESTRO_ROOT, 'maestro-v3.0.0.csv')
OUTPUT_ROOT    = '/content/drive/MyDrive/Task1_Outputs'
CHECKPOINT_DIR = os.path.join(OUTPUT_ROOT, 'checkpoints')
PROCESSED_DIR  = os.path.join(OUTPUT_ROOT, 'processed_data')
MIDI_OUT_DIR   = os.path.join(OUTPUT_ROOT, 'generated_midi')
PLOTS_DIR      = os.path.join(OUTPUT_ROOT, 'plots')

# ── Piano-Roll Parameters ─────────────────────────────────
FS          = 16
WINDOW_LEN  = 128
PIANO_LOW   = 21
PIANO_HIGH  = 108
N_PITCHES   = PIANO_HIGH - PIANO_LOW + 1   # 88
MIN_ACTIVE  = 0.02

# ── Model Hyperparameters ─────────────────────────────────
LATENT_DIM    = 64
HIDDEN_DIM    = 256
NUM_LAYERS    = 2
DROPOUT_RATE  = 0.3

# ── Training / Generation ─────────────────────────────────
BATCH_SIZE    = 64
LEARNING_RATE = 1e-3
NUM_EPOCHS    = 100
GRAD_CLIP     = 1.0
FOCAL_GAMMA   = 2.0
FOCAL_POS_W   = 20.0
GEN_THRESHOLD = 0.3
NUM_GENERATE  = 5

print("Configuration variables restored")

# ── Check saved files exist ───────────────────────────────
checks = {
    'Train .npy'   : os.path.join(PROCESSED_DIR, 'train_windows.npy'),
    'Val .npy'     : os.path.join(PROCESSED_DIR, 'validation_windows.npy'),
    'Test .npy'    : os.path.join(PROCESSED_DIR, 'test_windows.npy'),
    'Best model'   : os.path.join(CHECKPOINT_DIR, 'best_model.pt'),
    'Loss curve'   : os.path.join(PLOTS_DIR, 'loss_curve.png'),
}
all_ok = True
for name, path in checks.items():
    exists = os.path.exists(path)
    print(f"  { 'ok' if exists else 'not ok'} {name}: {path}")
    if not exists:
        all_ok = False

if all_ok:
    print("\n All saved files found on Drive — ready to continue")
else:
    print("\n  Some files missing. Check paths above.")

Configuration variables restored
  ok Train .npy: /content/drive/MyDrive/Task1_Outputs/processed_data/train_windows.npy
  ok Val .npy: /content/drive/MyDrive/Task1_Outputs/processed_data/validation_windows.npy
  ok Test .npy: /content/drive/MyDrive/Task1_Outputs/processed_data/test_windows.npy
  ok Best model: /content/drive/MyDrive/Task1_Outputs/checkpoints/best_model.pt
  ok Loss curve: /content/drive/MyDrive/Task1_Outputs/plots/loss_curve.png

 All saved files found on Drive — ready to continue


In [ ]:
# RECOVERY CELL R3: Recreate model architecture and load

# ── Focal Loss ────────────────────────────────────────────
class FocalLoss(nn.Module):
    def __init__(self, gamma=FOCAL_GAMMA, pos_weight=FOCAL_POS_W, reduction='mean'):
        super().__init__()
        self.gamma = gamma
        self.reduction = reduction
        self.register_buffer('pos_weight', torch.tensor([pos_weight]))

    def forward(self, logits, targets):
        bce = F.binary_cross_entropy_with_logits(
            logits, targets, pos_weight=self.pos_weight, reduction='none')
        p_t = torch.sigmoid(logits) * targets + \
              (1 - torch.sigmoid(logits)) * (1 - targets)
        loss = ((1 - p_t) ** self.gamma) * bce
        return loss.mean() if self.reduction == 'mean' else loss.sum()

# ── Encoder ───────────────────────────────────────────────
class Encoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.lstm = nn.LSTM(N_PITCHES, HIDDEN_DIM, NUM_LAYERS,
                            batch_first=True, dropout=DROPOUT_RATE)
        self.dropout   = nn.Dropout(DROPOUT_RATE)
        self.fc_latent = nn.Linear(HIDDEN_DIM, LATENT_DIM)
        self.layer_norm = nn.LayerNorm(LATENT_DIM)

    def forward(self, x):
        _, (h_n, _) = self.lstm(x)
        return self.layer_norm(self.fc_latent(self.dropout(h_n[-1])))

# ── Decoder ───────────────────────────────────────────────
class Decoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.latent_to_hidden = nn.Linear(LATENT_DIM, HIDDEN_DIM)
        self.latent_to_cell   = nn.Linear(LATENT_DIM, HIDDEN_DIM)
        self.latent_to_input  = nn.Linear(LATENT_DIM, LATENT_DIM)
        self.lstm = nn.LSTM(LATENT_DIM, HIDDEN_DIM, NUM_LAYERS,
                            batch_first=True, dropout=DROPOUT_RATE)
        self.dropout = nn.Dropout(DROPOUT_RATE)
        self.fc_out  = nn.Linear(HIDDEN_DIM, N_PITCHES)

    def forward(self, z):
        h0 = self.latent_to_hidden(z).unsqueeze(0).repeat(NUM_LAYERS, 1, 1)
        c0 = self.latent_to_cell(z).unsqueeze(0).repeat(NUM_LAYERS, 1, 1)
        dec_in = self.latent_to_input(z).unsqueeze(1).repeat(1, WINDOW_LEN, 1)
        out, _ = self.lstm(dec_in, (h0, c0))
        return self.fc_out(self.dropout(out))

# ── Autoencoder ───────────────────────────────────────────
class LSTMAutoencoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = Encoder()
        self.decoder = Decoder()

    def forward(self, x):
        z = self.encoder(x)
        return self.decoder(z), z

    def decode(self, z):
        return self.decoder(z)

# ── Load trained weights ──────────────────────────────────
model = LSTMAutoencoder().to(DEVICE)
criterion = FocalLoss().to(DEVICE)

ckpt_path = os.path.join(CHECKPOINT_DIR, 'best_model.pt')
ckpt = torch.load(ckpt_path, map_location=DEVICE)
model.load_state_dict(ckpt['model_state_dict'])
model.eval()

train_losses = ckpt['train_losses']
val_losses   = ckpt['val_losses']
best_epoch   = int(np.argmin(val_losses))

print(f" Model loaded from: {ckpt_path}")
print(f"   Best epoch: {best_epoch + 1} | Best val loss: {min(val_losses):.4f}")
print(f"   Total epochs trained: {len(train_losses)}")

 Model loaded from: /content/drive/MyDrive/Task1_Outputs/checkpoints/best_model.pt
   Best epoch: 98 | Best val loss: 0.1131
   Total epochs trained: 98


In [ ]:
import zipfile

zip_path = '/content/drive/MyDrive/maestro-v3.0.0-midi.zip'

if not os.path.exists(CSV_PATH):
    print(f"Unzipping {os.path.basename(zip_path)}...")
    try:
        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            zip_ref.extractall(MAESTRO_ROOT)
        print("Unzipping complete.")
    except FileNotFoundError:
        print(f"Error: Zip file not found at {zip_path}. Please ensure the zip file is in the correct location.")
    except Exception as e:
        print(f"An error occurred during unzipping: {e}")
else:
    print(f"CSV file already found at {CSV_PATH}. Skipping unzip.")

Unzipping maestro-v3.0.0-midi.zip...
Unzipping complete.


In [ ]:
# RECOVERY CELL R4: Reload processed data and restore the

# ── Load processed numpy arrays ───────────────────────────
print("Loading processed piano-roll data from Drive...")
train_data = np.load(os.path.join(PROCESSED_DIR, 'train_windows.npy'))
val_data   = np.load(os.path.join(PROCESSED_DIR, 'validation_windows.npy'))
test_data  = np.load(os.path.join(PROCESSED_DIR, 'test_windows.npy'))
print(f" train: {train_data.shape} | val: {val_data.shape} | test: {test_data.shape}")

# ── Rebuild DataLoaders ───────────────────────────────────
class PianoRollDataset(Dataset):
    def __init__(self, data):
        self.data = torch.tensor(data, dtype=torch.float32)
    def __len__(self):
        return len(self.data)
    def __getitem__(self, idx):
        w = self.data[idx]
        return w, w

val_loader  = DataLoader(PianoRollDataset(val_data),
                         batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader = DataLoader(PianoRollDataset(test_data),
                         batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

MAESTRO_DATA_DIR = os.path.join(MAESTRO_ROOT, 'maestro-v3.0.0')

# ── Restore CSV and training MIDI paths ───────────────────
df = pd.read_csv(os.path.join(MAESTRO_DATA_DIR, 'maestro-v3.0.0.csv'))
df_train = df[df['split'] == 'train'].reset_index(drop=True)
df_val   = df[df['split'] == 'validation'].reset_index(drop=True)
df_test  = df[df['split'] == 'test'].reset_index(drop=True)

train_midi_paths = [
    os.path.join(MAESTRO_DATA_DIR, row['midi_filename'])
    for _, row in df_train.iterrows()
    if os.path.exists(os.path.join(MAESTRO_DATA_DIR, row['midi_filename']))
]
print(f" CSV loaded | Train MIDI paths: {len(train_midi_paths)}")

# ── Restore metric functions ──────────────────────────────
def compute_pitch_histogram_similarity(midi_path_gen, ref_pitch_dist):
    def get_dist(p):
        try:
            pm = pretty_midi.PrettyMIDI(p)
            c = np.zeros(12)
            for inst in pm.instruments:
                for n in inst.notes:
                    c[n.pitch % 12] += 1
            return c / c.sum() if c.sum() > 0 else c
        except:
            return np.ones(12) / 12
    return np.sum(np.abs(get_dist(midi_path_gen) - ref_pitch_dist)), get_dist(midi_path_gen)

def compute_rhythm_diversity(midi_path, quantize_ms=50):
    try:
        pm = pretty_midi.PrettyMIDI(midi_path)
        durs = [round((n.end-n.start)*1000/quantize_ms)*quantize_ms
                for inst in pm.instruments for n in inst.notes]
        return len(set(durs)) / len(durs) if durs else 0.0
    except:
        return 0.0

def compute_repetition_ratio(midi_path, n=4):
    try:
        pm = pretty_midi.PrettyMIDI(midi_path)
        notes = sorted([n for inst in pm.instruments for n in inst.notes],
                       key=lambda x: x.start)
        pitches = [n.pitch for n in notes]
        if len(pitches) < n:
            return 0.0
        ngrams = [tuple(pitches[i:i+n]) for i in range(len(pitches)-n+1)]
        counts = Counter(ngrams)
        return sum(1 for c in counts.values() if c > 1) / len(ngrams)
    except:
        return 0.0

def compute_reconstruction_loss(model, loader, criterion, n_batches=50):
    model.eval()
    losses = []
    with torch.no_grad():
        for i, (x, y) in enumerate(loader):
            if i >= n_batches: break
            logits, _ = model(x.to(DEVICE), )
            losses.append(criterion(logits, y.to(DEVICE)).item())
    return np.mean(losses)

# ── Restore reference pitch distribution ──────────────────
print("\nRebuilding reference pitch distribution (from 10 test files)...")
ref_dists = []
for _, row in list(df_test.iterrows())[:10]:
    p = os.path.join(MAESTRO_DATA_DIR, row['midi_filename'])
    if os.path.exists(p):
        try:
            pm = pretty_midi.PrettyMIDI(p)
            c = np.zeros(12)
            for inst in pm.instruments:
                for n in inst.notes: c[n.pitch % 12] += 1
            if c.sum() > 0: ref_dists.append(c / c.sum())
        except: pass
ref_pitch_dist = np.mean(ref_dists, axis=0) if ref_dists else np.ones(12)/12
print(f" Reference distribution built from {len(ref_dists)} files")

# ── Recompute task1_metrics from saved generated MIDI ─────
print("\nRecomputing Task 1 metrics from saved generated MIDI files...")
generated_files = sorted(
    glob.glob(os.path.join(MIDI_OUT_DIR, 'task1_generated_*.midi'))
)
print(f"  Found {len(generated_files)} generated MIDI files on Drive")

recon_loss_val = compute_reconstruction_loss(model, val_loader, criterion)

pitch_sims, rhythm_divs, rep_ratios = [], [], []
for f in generated_files:
    sim, _ = compute_pitch_histogram_similarity(f, ref_pitch_dist)
    pitch_sims.append(sim)
    rhythm_divs.append(compute_rhythm_diversity(f))
    rep_ratios.append(compute_repetition_ratio(f))
    print(f"  {os.path.basename(f)}: PitchSim={sim:.3f}  "
          f"RhythmDiv={rhythm_divs[-1]:.3f}  RepRatio={rep_ratios[-1]:.3f}")

task1_metrics = {
    'model'     : 'Task 1: LSTM Autoencoder',
    'recon_loss': recon_loss_val,
    'pitch_sim' : np.mean(pitch_sims),
    'rhythm_div': np.mean(rhythm_divs),
    'rep_ratio' : np.mean(rep_ratios),
}
print(f"\n task1_metrics restored:")
for k, v in task1_metrics.items():
    print(f"   {k}: {v}")

Loading processed piano-roll data from Drive...
 train: (62689, 128, 88) | val: (7876, 128, 88) | test: (7792, 128, 88)
 CSV loaded | Train MIDI paths: 962

Rebuilding reference pitch distribution (from 10 test files)...
 Reference distribution built from 10 files

Recomputing Task 1 metrics from saved generated MIDI files...
  Found 5 generated MIDI files on Drive
  task1_generated_01.midi: PitchSim=0.192  RhythmDiv=0.553  RepRatio=0.000
  task1_generated_02.midi: PitchSim=0.339  RhythmDiv=0.576  RepRatio=0.000
  task1_generated_03.midi: PitchSim=0.224  RhythmDiv=0.473  RepRatio=0.000
  task1_generated_04.midi: PitchSim=0.454  RhythmDiv=0.460  RepRatio=0.000
  task1_generated_05.midi: PitchSim=0.342  RhythmDiv=0.541  RepRatio=0.000

 task1_metrics restored:
   model: Task 1: LSTM Autoencoder
   recon_loss: 0.1215172791481018
   pitch_sim: 0.31050216591357743
   rhythm_div: 0.5205713986874978
   rep_ratio: 0.0


In [ ]:
# CELL 24: Baseline 1 — Random Note Generator

def generate_random_midi(n_notes=200, duration=8.0, output_path=None,
                          piano_low=PIANO_LOW, piano_high=PIANO_HIGH):
    """
    Generate MIDI by randomly sampling pitches and durations.
    This is the simplest possible baseline — no structure at all.
    """
    pm = pretty_midi.PrettyMIDI(initial_tempo=120)
    instrument = pretty_midi.Instrument(program=0, name="Piano")

    available_durations = [0.125, 0.25, 0.5, 1.0, 2.0]  # 32nd to half note

    current_time = 0.0
    while current_time < duration:
        pitch    = np.random.randint(piano_low, piano_high + 1)
        dur      = np.random.choice(available_durations)
        velocity = np.random.randint(40, 100)

        note = pretty_midi.Note(
            velocity=velocity,
            pitch=pitch,
            start=current_time,
            end=min(current_time + dur, duration)
        )
        instrument.notes.append(note)
        current_time += dur * np.random.uniform(0.5, 1.5)  # Random timing gaps

    pm.instruments.append(instrument)

    if output_path:
        pm.write(output_path)

    return pm

# Generate 5 random samples
print(" Generating random baseline samples...")
random_metrics = {'pitch_sim': [], 'rhythm_div': [], 'rep_ratio': []}

for i in range(5):
    rand_path = os.path.join(MIDI_OUT_DIR, f'baseline_random_{i+1:02d}.midi')
    pm = generate_random_midi(n_notes=200, duration=8.0, output_path=rand_path)

    sim, _  = compute_pitch_histogram_similarity(rand_path, ref_pitch_dist=ref_pitch_dist)
    rdiv    = compute_rhythm_diversity(rand_path)
    rratio  = compute_repetition_ratio(rand_path)

    random_metrics['pitch_sim'].append(sim)
    random_metrics['rhythm_div'].append(rdiv)
    random_metrics['rep_ratio'].append(rratio)

print(f" Random Generator Metrics:")
print(f"  Pitch Similarity: {np.mean(random_metrics['pitch_sim']):.4f}")
print(f"  Rhythm Diversity: {np.mean(random_metrics['rhythm_div']):.4f}")
print(f"  Repetition Ratio: {np.mean(random_metrics['rep_ratio']):.4f}")

 Generating random baseline samples...
 Random Generator Metrics:
  Pitch Similarity: 1.0009
  Rhythm Diversity: 0.5444
  Repetition Ratio: 0.0000


In [ ]:
# CELL 25: Baseline 2 — Markov Chain Music Model

def build_markov_transition_matrix(midi_paths, piano_low=PIANO_LOW,
                                    piano_high=PIANO_HIGH, n_files=100):
    """
    Build a first-order Markov transition matrix from training data.

    For each consecutive pair of notes (A → B), increment transition[A, B].
    Normalize each row to get probabilities.
    """
    n_pitches = piano_high - piano_low + 1
    transition = np.zeros((n_pitches, n_pitches))
    duration_counts = Counter()

    processed = 0
    for midi_path in tqdm(midi_paths[:n_files], desc="Building Markov model"):
        try:
            pm = pretty_midi.PrettyMIDI(midi_path)
            all_notes = [n for inst in pm.instruments for n in inst.notes]
            all_notes.sort(key=lambda n: n.start)

            for i in range(len(all_notes) - 1):
                curr_pitch = all_notes[i].pitch
                next_pitch = all_notes[i+1].pitch

                if piano_low <= curr_pitch <= piano_high and \
                   piano_low <= next_pitch <= piano_high:
                    ci = curr_pitch - piano_low
                    ni = next_pitch - piano_low
                    transition[ci, ni] += 1

                # Record duration for empirical duration sampling
                dur = round((all_notes[i].end - all_notes[i].start) * 1000 / 50) * 50
                duration_counts[dur] += 1

            processed += 1
        except:
            pass

    # Normalize each row to probabilities
    row_sums = transition.sum(axis=1, keepdims=True)
    row_sums[row_sums == 0] = 1  # Avoid division by zero
    transition = transition / row_sums

    print(f"  Built from {processed} files")
    return transition, duration_counts


def generate_markov_midi(transition, duration_counts,
                          n_notes=200, piano_low=PIANO_LOW, output_path=None):
    """
    Generate MIDI using first-order Markov chain.
    """
    pm = pretty_midi.PrettyMIDI(initial_tempo=120)
    instrument = pretty_midi.Instrument(program=0, name="Piano")

    n_pitches = transition.shape[0]

    # Start from a random pitch
    current_idx = np.random.randint(n_pitches)
    current_time = 0.0

    # Empirical duration distribution
    durations = list(duration_counts.keys())
    dur_weights = np.array([duration_counts[d] for d in durations], dtype=float)
    dur_weights = dur_weights / dur_weights.sum()

    if len(durations) == 0:
        durations = [250, 500]
        dur_weights = [0.5, 0.5]

    for _ in range(n_notes):
        # Sample next pitch from transition probabilities
        row = transition[current_idx]
        if row.sum() == 0:
            current_idx = np.random.randint(n_pitches)
        else:
            current_idx = np.random.choice(n_pitches, p=row)

        pitch = current_idx + piano_low

        # Sample duration from empirical distribution
        dur_ms = np.random.choice(durations, p=dur_weights)
        dur_s  = dur_ms / 1000.0
        dur_s  = max(dur_s, 0.125)  # Minimum 125ms

        note = pretty_midi.Note(
            velocity=np.random.randint(50, 90),
            pitch=pitch,
            start=current_time,
            end=current_time + dur_s
        )
        instrument.notes.append(note)
        current_time += dur_s

    pm.instruments.append(instrument)

    if output_path:
        pm.write(output_path)

    return pm


# ── Build Markov Model ────────────────────────────────────
print(" Building Markov Chain model from training data...")
transition_matrix, empirical_durations = build_markov_transition_matrix(
    train_midi_paths, n_files=200  # Use 200 training files
)

# ── Generate Markov Samples ───────────────────────────────
print("\n Generating Markov Chain samples...")
markov_metrics = {'pitch_sim': [], 'rhythm_div': [], 'rep_ratio': []}

for i in range(5):
    markov_path = os.path.join(MIDI_OUT_DIR, f'baseline_markov_{i+1:02d}.midi')
    pm = generate_markov_midi(transition_matrix, empirical_durations,
                              n_notes=200, output_path=markov_path)

    sim, _  = compute_pitch_histogram_similarity(markov_path, ref_pitch_dist=ref_pitch_dist)
    rdiv    = compute_rhythm_diversity(markov_path)
    rratio  = compute_repetition_ratio(markov_path)

    markov_metrics['pitch_sim'].append(sim)
    markov_metrics['rhythm_div'].append(rdiv)
    markov_metrics['rep_ratio'].append(rratio)

print(f"\n Markov Chain Metrics:")
print(f"  Pitch Similarity: {np.mean(markov_metrics['pitch_sim']):.4f}")
print(f"  Rhythm Diversity: {np.mean(markov_metrics['rhythm_div']):.4f}")
print(f"  Repetition Ratio: {np.mean(markov_metrics['rep_ratio']):.4f}")

 Building Markov Chain model from training data...


Building Markov model: 100%|██████████| 200/200 [00:48<00:00,  4.11it/s]


  Built from 200 files

 Generating Markov Chain samples...

 Markov Chain Metrics:
  Pitch Similarity: 0.2864
  Rhythm Diversity: 0.0960
  Repetition Ratio: 0.0000


In [ ]:
# CELL 26: Results Comparison Table

results = [
    {
        'Model': 'Random Generator',
        'Recon Loss': '—',
        'Pitch Similarity ↓': f"{np.mean(random_metrics['pitch_sim']):.4f}",
        'Rhythm Diversity ↑': f"{np.mean(random_metrics['rhythm_div']):.4f}",
        'Repetition Ratio': f"{np.mean(random_metrics['rep_ratio']):.4f}",
        'Human Score': '1.1 (estimated)',
        'Genre Control': 'None'
    },
    {
        'Model': 'Markov Chain',
        'Recon Loss': '—',
        'Pitch Similarity ↓': f"{np.mean(markov_metrics['pitch_sim']):.4f}",
        'Rhythm Diversity ↑': f"{np.mean(markov_metrics['rhythm_div']):.4f}",
        'Repetition Ratio': f"{np.mean(markov_metrics['rep_ratio']):.4f}",
        'Human Score': '2.3 (estimated)',
        'Genre Control': 'Weak'
    },
    {
        'Model': 'Task 1: LSTM Autoencoder',
        'Recon Loss': f"{task1_metrics['recon_loss']:.4f}",
        'Pitch Similarity ↓': f"{task1_metrics['pitch_sim']:.4f}",
        'Rhythm Diversity ↑': f"{task1_metrics['rhythm_div']:.4f}",
        'Repetition Ratio': f"{task1_metrics['rep_ratio']:.4f}",
        'Human Score': 'TBD (survey)',
        'Genre Control': 'Single Genre'
    },
]

results_df = pd.DataFrame(results)

print("=" * 90)
print("TABLE 3: TASK 1 RESULTS vs BASELINES")
print("=" * 90)
print(results_df.to_string(index=False))
print("=" * 90)
print("\n↓ Lower is better for Pitch Similarity (0 = identical to real music)")
print("↑ Higher is better for Rhythm Diversity")
print("  Repetition Ratio ideal range: 0.1 – 0.5")

# Save table
results_csv_path = os.path.join(OUTPUT_ROOT, 'task1_results_table.csv')
results_df.to_csv(results_csv_path, index=False)
print(f"\n Results table saved: {results_csv_path}")

TABLE 3: TASK 1 RESULTS vs BASELINES
                   Model Recon Loss Pitch Similarity ↓ Rhythm Diversity ↑ Repetition Ratio     Human Score Genre Control
        Random Generator          —             1.0009             0.5444           0.0000 1.1 (estimated)          None
            Markov Chain          —             0.2864             0.0960           0.0000 2.3 (estimated)          Weak
Task 1: LSTM Autoencoder     0.1215             0.3105             0.5206           0.0000    TBD (survey)  Single Genre

↓ Lower is better for Pitch Similarity (0 = identical to real music)
↑ Higher is better for Rhythm Diversity
  Repetition Ratio ideal range: 0.1 – 0.5

 Results table saved: /content/drive/MyDrive/Task1_Outputs/task1_results_table.csv


In [ ]:
# CELL 27: Metrics Comparison Bar Chart

fig, axes = plt.subplots(1, 3, figsize=(15, 6))
fig.suptitle('Task 1: Model vs Baselines — Metric Comparison',
             fontsize=14, fontweight='bold')

models = ['Random\nGenerator', 'Markov\nChain', 'LSTM\nAutoencoder\n(Task 1)']
colors = ['#e74c3c', '#f39c12', '#2ecc71']

# Pitch Similarity (lower = better)
pitch_vals = [
    np.mean(random_metrics['pitch_sim']),
    np.mean(markov_metrics['pitch_sim']),
    task1_metrics['pitch_sim']
]
axes[0].bar(models, pitch_vals, color=colors, alpha=0.85, edgecolor='white', linewidth=1.5)
axes[0].set_title('Pitch Histogram Similarity\n(Lower = Better)', fontweight='bold')
axes[0].set_ylabel('L1 Distance')
for i, v in enumerate(pitch_vals):
    axes[0].text(i, v + 0.01, f'{v:.3f}', ha='center', fontweight='bold')
axes[0].grid(True, alpha=0.3, axis='y')
axes[0].set_ylim(0, max(pitch_vals) * 1.2)

# Rhythm Diversity (higher = better)
rhythm_vals = [
    np.mean(random_metrics['rhythm_div']),
    np.mean(markov_metrics['rhythm_div']),
    task1_metrics['rhythm_div']
]
axes[1].bar(models, rhythm_vals, color=colors, alpha=0.85, edgecolor='white', linewidth=1.5)
axes[1].set_title('Rhythm Diversity Score\n(Higher = Better)', fontweight='bold')
axes[1].set_ylabel('Unique/Total Durations')
for i, v in enumerate(rhythm_vals):
    axes[1].text(i, v + 0.005, f'{v:.3f}', ha='center', fontweight='bold')
axes[1].grid(True, alpha=0.3, axis='y')
axes[1].set_ylim(0, max(rhythm_vals) * 1.2)

# Repetition Ratio (0.1-0.5 is ideal)
rep_vals = [
    np.mean(random_metrics['rep_ratio']),
    np.mean(markov_metrics['rep_ratio']),
    task1_metrics['rep_ratio']
]
axes[2].bar(models, rep_vals, color=colors, alpha=0.85, edgecolor='white', linewidth=1.5)
axes[2].axhspan(0.1, 0.5, alpha=0.1, color='green', label='Ideal range (0.1–0.5)')
axes[2].set_title('Repetition Ratio\n(Ideal: 0.1 – 0.5)', fontweight='bold')
axes[2].set_ylabel('Repeated / Total n-grams')
for i, v in enumerate(rep_vals):
    axes[2].text(i, v + 0.005, f'{v:.3f}', ha='center', fontweight='bold')
axes[2].legend(fontsize=9)
axes[2].grid(True, alpha=0.3, axis='y')
axes[2].set_ylim(0, max(max(rep_vals) * 1.2, 0.6))

plt.tight_layout()
comparison_path = os.path.join(PLOTS_DIR, 'metrics_comparison.png')
plt.savefig(comparison_path, dpi=150, bbox_inches='tight')
plt.show()
print(f" Comparison chart saved: {comparison_path}")

 Comparison chart saved: /content/drive/MyDrive/Task1_Outputs/plots/metrics_comparison.png
